In [2]:
import polars as pl

# 1. Load our engineered features at lightning speed
file_path = "data/AMZN_engineered_features.parquet"
df = pl.read_parquet(file_path)

# Drop any rows where our target is null (the very end of the day)
df = df.drop_nulls()

# 2. Chronological Train/Test Split (80% Train, 20% Test)
split_idx = int(df.height * 0.8)

# The first 80% of rows
df_train = df.head(split_idx)
# The remaining 20% of rows
df_test = df.tail(df.height - split_idx)

# Let's verify the split
print(f"Total Events: {df.height}")
print(f"Training Events (Morning/Mid-day): {df_train.height}")
print(f"Testing Events (Afternoon): {df_test.height}")

Total Events: 269728
Training Events (Morning/Mid-day): 215782
Testing Events (Afternoon): 53946


In [3]:
# Define our predictive features (X) and our target (y)
features = ["Spread_Dollars", "Micro_Price_Dollars", "Imbalance_Ratio", "Deep_Imbalance_Ratio"]
target = "Target_Direction"

# Convert Polars dataframes to NumPy arrays for Scikit-Learn
X_train = df_train.select(features).to_numpy()
y_train = df_train.select(target).to_numpy().flatten()

X_test = df_test.select(features).to_numpy()
y_test = df_test.select(target).to_numpy().flatten()

print("Data matrices prepared for machine learning!")

Data matrices prepared for machine learning!


In [5]:
!pip install scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 2.2 MB/s eta 0:00:000:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 1.8 MB/s eta 0:00:00m eta 0:00:010:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn]0m 5/6 [scikit-learn]]


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. Initialize the AI (Baseline Random Forest)
rf_model = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)

# 2. Train the model on the morning data
print("Training the Random Forest model... (this might take 5 to 15 seconds)")
rf_model.fit(X_train, y_train)

# 3. Predict the afternoon data (The Blind Test)
print("Generating predictions on unseen afternoon data...")
y_pred = rf_model.predict(X_test)

# 4. Score the model
print("\n--- Model Performance on Afternoon Test Data ---")
print(classification_report(y_test, y_pred, target_names=["Down (-1)", "Flat (0)", "Up (1)"]))

Training the Random Forest model... (this might take 5 to 15 seconds)
Generating predictions on unseen afternoon data...

--- Model Performance on Afternoon Test Data ---
              precision    recall  f1-score   support

   Down (-1)       0.45      0.31      0.37     18910
    Flat (0)       0.32      0.28      0.30     16845
      Up (1)       0.40      0.57      0.47     18191

    accuracy                           0.39     53946
   macro avg       0.39      0.39      0.38     53946
weighted avg       0.39      0.39      0.38     53946



In [7]:
import numpy as np

# Extract importance scores
importances = rf_model.feature_importances_

# Sort them from most important to least important
indices = np.argsort(importances)[::-1]

print("--- AI Feature Importance ---")
for i in indices:
    print(f"{features[i]}: {importances[i]:.2%}")

--- AI Feature Importance ---
Micro_Price_Dollars: 32.95%
Imbalance_Ratio: 32.86%
Spread_Dollars: 18.25%
Deep_Imbalance_Ratio: 15.94%


In [3]:
# Calculate momentum (change over the last 10 events)
df = df.with_columns([
    (pl.col("Micro_Price_Dollars") - pl.col("Micro_Price_Dollars").shift(10)).alias("Micro_Price_Delta_10"),
    (pl.col("Deep_Imbalance_Ratio") - pl.col("Deep_Imbalance_Ratio").shift(10)).alias("Deep_Imbalance_Delta_10")
])

# Shifting backwards creates 'null' values for the very first 10 rows of the day (since there is no past).
df = df.drop_nulls()

# Let's inspect our new velocity metrics!
columns_to_view = [
    "Micro_Price_Dollars", "Micro_Price_Delta_10", 
    "Deep_Imbalance_Ratio", "Deep_Imbalance_Delta_10"
]
print(df.select(columns_to_view).head(10))

shape: (10, 4)
┌─────────────────────┬──────────────────────┬──────────────────────┬─────────────────────────┐
│ Micro_Price_Dollars ┆ Micro_Price_Delta_10 ┆ Deep_Imbalance_Ratio ┆ Deep_Imbalance_Delta_10 │
│ ---                 ┆ ---                  ┆ ---                  ┆ ---                     │
│ f64                 ┆ f64                  ┆ f64                  ┆ f64                     │
╞═════════════════════╪══════════════════════╪══════════════════════╪═════════════════════════╡
│ 223.834298          ┆ 0.269298             ┆ 0.316681             ┆ -0.545585               │
│ 223.834298          ┆ 0.0                  ┆ 0.301903             ┆ -0.5594                 │
│ 223.834298          ┆ 0.0                  ┆ 0.295512             ┆ -0.571933               │
│ 223.834298          ┆ 0.0                  ┆ 0.283509             ┆ -0.583936               │
│ 223.834298          ┆ 0.0                  ┆ 0.28282              ┆ -0.583621               │
│ 223.834298          ┆ 0

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. Update our feature list to include the new momentum metrics
features = [
    "Spread_Dollars", "Micro_Price_Dollars", "Imbalance_Ratio", 
    "Deep_Imbalance_Ratio", "Micro_Price_Delta_10", "Deep_Imbalance_Delta_10"
]
target = "Target_Direction"

# 2. Re-split the data chronologically (since our row count changed)
split_idx = int(df.height * 0.8)
df_train = df.head(split_idx)
df_test = df.tail(df.height - split_idx)

# Convert to NumPy
X_train = df_train.select(features).to_numpy()
y_train = df_train.select(target).to_numpy().flatten()
X_test = df_test.select(features).to_numpy()
y_test = df_test.select(target).to_numpy().flatten()

# 3. Train the Upgraded AI
print("Training Upgraded AI with Momentum... (5 to 15 seconds)")
rf_model_v2 = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf_model_v2.fit(X_train, y_train)

# 4. Score the upgraded model
y_pred_v2 = rf_model_v2.predict(X_test)
print("\n--- Upgraded Model Performance ---")
print(classification_report(y_test, y_pred_v2, target_names=["Down (-1)", "Flat (0)", "Up (1)"]))

# 5. Check what features it cares about now
import numpy as np
importances = rf_model_v2.feature_importances_
indices = np.argsort(importances)[::-1]

print("\n--- Upgraded AI Feature Importance ---")
for i in indices:
    print(f"{features[i]}: {importances[i]:.2%}")